In [1]:
import pandas as pd
import numpy as np
import random

In [7]:
def generate_dataset(rows=100, seed=42):
    random.seed(seed)
    np.random.seed(seed)

    data = []

    districts = [
        "Andheri", "Ambernath", "Dadar",
        "Thane", "Virar",
        "Vashi", "Nerul",
        "Panvel", "Dombivli", "Mira Road"
    ]

    seasons = ["Summer", "Monsoon", "Winter", "Festival"]

    jobs = [
        "Electrician", "Plumber", "Carpenter", "Painter",
        "Appliance Technician", "House Cleaner",
        "Driver", "Cook", "Mechanic", "Masonry"
    ]

    for _ in range(rows):

        district = random.choice(districts)
        season = random.choice(seasons)
        jobType = random.choice(jobs)

        # Multi-skilled worker
        isMultiTalented = random.choice([0, 1])

        # Years of experience
        experienceYears = random.randint(1, 15)

        # Jobs completed till date
        jobsCompleted = random.randint(5, 100)

        # --------------------------------
        # Seasonal Demand Logic
        # --------------------------------

        demand_score = 0.2  # base demand

        # Seasonal boosting logic
        if season == "Monsoon" and jobType in ["Plumber", "Electrician", "Masonry"]:
            demand_score += 0.35

        if season == "Summer" and jobType in ["Appliance Technician", "Electrician"]:
            demand_score += 0.30

        if season == "Festival" and jobType in ["Painter", "Carpenter", "House Cleaner"]:
            demand_score += 0.40

        if season == "Winter" and jobType in ["Cook", "Driver"]:
            demand_score += 0.20

        # Experience increases reliability
        if experienceYears > 7:
            demand_score += 0.10

        # Multi-skilled workers get more jobs
        if isMultiTalented == 1:
            demand_score += 0.15

        # Past completed jobs influence repeat demand
        if jobsCompleted > 500:
            demand_score += 0.10

        # Controlled randomness
        demand_score += random.uniform(-0.05, 0.05)

        # Final label: High Seasonal Demand
        highDemandNextSeason = 1 if demand_score > 0.5 else 0

        data.append([
            district,
            season,
            jobType,
            jobsCompleted,
            experienceYears,
            isMultiTalented,
            round(demand_score, 2),
            highDemandNextSeason
        ])

    columns = [
        "district",
        "season",
        "jobType",
        "jobsCompleted",
        "experienceYears",
        "isMultiTalented",
        "seasonalDemandScore",
        "highDemandNextSeason"
    ]

    df = pd.DataFrame(data, columns=columns)

    return df

In [ ]:
df = generate_dataset(rows=100)

df.head(20)

,district,season,jobType,jobsCompleted,experienceYears,isMultiTalented,seasonalDemandScore,highDemandNextSeason
0,Ambernath,Summer,Appliance Technician,162,4,0,0.52,1
1,Dombivli,Summer,Masonry,50,1,1,0.31,0
2,Thane,Summer,Mechanic,685,12,0,0.42,0
3,Nerul,Monsoon,Cook,910,13,1,0.50,1
4,Dadar,Festival,House Cleaner,240,3,1,0.80,1
5,Vashi,Summer,Plumber,387,2,1,0.38,0
6,Mira Road,Winter,Electrician,147,9,1,0.50,0
7,Nerul,Summer,Mechanic,663,14,1,0.56,1
8,Vashi,Monsoon,Plumber,253,11,0,0.68,1
9,Ambernath,Monsoon,Plumber,484,5,1,0.71,1


In [8]:
df.describe(include="all")
df["highDemandNextSeason"].value_counts(normalize=True)

highDemandNextSeason
0    0.66
1    0.34
Name: proportion, dtype: float64

In [9]:
df.to_csv("voicesewa_seasonal_worker_dataset.csv", index=False)
print("Dataset saved successfully!")

Dataset saved successfully!


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Target
y = df["highDemandNextSeason"]

# Drop target + seasonalDemandScore (avoid leakage)
X = df.drop(["highDemandNextSeason", "seasonalDemandScore"], axis=1)

# One-hot encoding categorical columns
X = pd.get_dummies(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Initialize model
model = RandomForestClassifier(random_state=42)

In [13]:
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [14]:
print(model.feature_names_in_)


['jobsCompleted' 'experienceYears' 'isMultiTalented' 'district_Ambernath'
 'district_Andheri' 'district_Dadar' 'district_Dombivli'
 'district_Mira Road' 'district_Nerul' 'district_Panvel' 'district_Thane'
 'district_Vashi' 'district_Virar' 'season_Festival' 'season_Monsoon'
 'season_Summer' 'season_Winter' 'jobType_Appliance Technician'
 'jobType_Carpenter' 'jobType_Cook' 'jobType_Driver' 'jobType_Electrician'
 'jobType_House Cleaner' 'jobType_Masonry' 'jobType_Mechanic'
 'jobType_Painter' 'jobType_Plumber']


In [15]:
import pickle

with open("voicesewa_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("Model saved successfully!")

Model saved successfully!


In [16]:
loaded_model = pickle.load(open("voicesewa_model.pkl", "rb"))

loaded_model.predict(X_test[:5])

array([1, 0, 0, 0, 0])